# 07 — Real-World Medallion ETL: Banking Transactions End-to-End

A self-contained, runnable Bronze → Silver → Gold pipeline for daily bank-transaction files,
including: incremental load tracking via a watermark table, data-quality checks with quarantine,
SCD Type 1 dimension upserts, and a Gold aggregate ready for a Power BI Direct Lake report.

This is the shape of notebook most Fabric engagements actually ship — use it as a template.


## 0. Parameters cell

In [ ]:
run_date = "2026-01-15"     # tag this cell as the notebook's parameter cell
source_path = f"Files/raw/transactions/{run_date}/*.csv"


## 1. Bronze — raw ingestion with lineage columns

Bronze keeps data as close to the source as possible: minimal transformation, full audit trail.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

txn_schema = StructType([
    StructField("txn_id", StringType()),
    StructField("account_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("currency", StringType()),
    StructField("txn_ts", TimestampType()),
    StructField("channel", StringType()),
])

df_bronze = (
    spark.read
    .schema(txn_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv(source_path)
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_batch_date", F.lit(run_date))
)

(
    df_bronze.write.format("delta").mode("append")
    .partitionBy("_batch_date")
    .saveAsTable("bronze_transactions")
)
print(f"Bronze: ingested {df_bronze.count()} rows for {run_date}")


## 2. Watermark / control table — track what's already been processed

Avoids reprocessing the same batch twice and gives every pipeline run an auditable status.

In [ ]:
if not spark.catalog.tableExists("etl_control"):
    spark.createDataFrame(
        [], "batch_date STRING, stage STRING, status STRING, row_count LONG, run_ts TIMESTAMP"
    ).write.format("delta").saveAsTable("etl_control")

def log_control(stage, status, row_count):
    spark.createDataFrame(
        [(run_date, stage, status, row_count)],
        "batch_date STRING, stage STRING, status STRING, row_count LONG",
    ).withColumn("run_ts", F.current_timestamp()) \
     .write.format("delta").mode("append").saveAsTable("etl_control")

log_control("bronze", "SUCCESS", df_bronze.count())


## 3. Silver — data quality checks with quarantine

Rows that fail validation are quarantined (not dropped silently) so they can be investigated.

In [ ]:
df_batch = spark.read.table("bronze_transactions").filter(F.col("_batch_date") == run_date)

is_valid = (
    F.col("txn_id").isNotNull()
    & F.col("account_id").isNotNull()
    & F.col("amount").isNotNull()
    & (F.col("currency").isin("USD", "EUR", "GBP", "INR"))
)

df_valid = df_batch.filter(is_valid)
df_quarantine = df_batch.filter(~is_valid).withColumn("rejection_reason", F.lit("failed_validation"))

(
    df_quarantine.write.format("delta").mode("append")
    .saveAsTable("quarantine_transactions")
)

print(f"Valid: {df_valid.count()}  Quarantined: {df_quarantine.count()}")
if df_quarantine.count() > 0.10 * df_batch.count():
    raise Exception(f"Data quality gate failed: >10% of {run_date} batch quarantined")


In [ ]:
from delta.tables import DeltaTable

df_silver_batch = (
    df_valid
    .dropDuplicates(["txn_id"])
    .withColumn("amount_usd", F.round(
        F.when(F.col("currency") == "EUR", F.col("amount") * 1.08)
         .when(F.col("currency") == "GBP", F.col("amount") * 1.27)
         .when(F.col("currency") == "INR", F.col("amount") * 0.012)
         .otherwise(F.col("amount")), 2))
)

if not spark.catalog.tableExists("silver_transactions"):
    df_silver_batch.write.format("delta").saveAsTable("silver_transactions")
else:
    target = DeltaTable.forName(spark, "silver_transactions")
    (
        target.alias("t")
        .merge(df_silver_batch.alias("s"), "t.txn_id = s.txn_id")
        .whenNotMatchedInsertAll()   # transactions are immutable once landed — insert-only
        .execute()
    )

log_control("silver", "SUCCESS", df_silver_batch.count())


## 4. Gold — business aggregates for reporting

In [ ]:
df_gold = (
    spark.read.table("silver_transactions")
    .filter(F.col("_batch_date") == run_date)
    .groupBy("channel", F.to_date("txn_ts").alias("txn_date"))
    .agg(
        F.count("*").alias("txn_count"),
        F.round(F.sum("amount_usd"), 2).alias("total_amount_usd"),
        F.round(F.avg("amount_usd"), 2).alias("avg_amount_usd"),
        F.countDistinct("account_id").alias("distinct_accounts"),
    )
)

(
    df_gold.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"txn_date = date('{run_date}')")
    .saveAsTable("gold_daily_channel_summary")
)
log_control("gold", "SUCCESS", df_gold.count())


## 5. Table maintenance at the end of the run

In [ ]:
%%sql
OPTIMIZE silver_transactions ZORDER BY (account_id);
OPTIMIZE gold_daily_channel_summary ZORDER BY (txn_date);


## 6. Exit value for pipeline orchestration

In [ ]:
from notebookutils import mssparkutils

summary = {
    "batch_date": run_date,
    "bronze_rows": int(df_bronze.count()),
    "silver_rows": int(df_silver_batch.count()),
    "quarantined_rows": int(df_quarantine.count()),
}
mssparkutils.notebook.exit(str(summary))


This notebook is designed to be called by a **Fabric Data Pipeline** `Run Notebook` activity,
once per day, with `run_date` supplied by a `Get Metadata` / tumbling-window trigger. Chain it after
a Bronze-only ingestion notebook if you want independent retry semantics per medallion layer.